In [ ]:
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt
from matplotlib import colors, cm

# Pipeline functions (Steps 0.5-7) live in pipeline.py, in this same folder.
# See that file for full docstrings/implementation details.
from pipeline import (
    clean_temperature_observations,
    get_sentinel2_bands,
    clip_bands_to_aoi,
    resample_all_bands_to_10m,
    dataarray_to_numpy,
    profile_from_dataarray,
    build_all_focal_buffers,
    build_training_table,
    train_rf_model,
    predict_temperature_surface,
)

In [ ]:
# Load your volunteer traverse observations as shapefiles
points = gpd.read_file('')

# STEP 0.5: Clean observations (you can customize the NAN values and valid temperature range) 
points = clean_temperature_observations(
    # NOTE: Update source column to match your data 
    # Output column is used in the model later, so update that code if you change it here
    points, source_col="Temp_F", output_col="temperature_f"
)

In [ ]:
# Define constants 

# Target date to search for satellite images around your campaign date
target_date = ""

# Bounding box determined from extent of measurements
bounding_box = points.geometry.total_bounds

# Define output path for model's final raster file output 
tif_output_path = ""

In [ ]:
# STEP 1: Fetch Sentinel 2 satellite bands

band_arrays = get_sentinel2_bands(bounding_box, target_date)

In [ ]:
# STEP 1b: Clip satellite bands to area of interest (AOI) defined by bounding box set above
clipped_bands = clip_bands_to_aoi(band_arrays, bounding_box, buffer_m = 1000)

# STEP 2: Resample satellite bands for model usage
# Resample all bands to 10m (as some bands are natively 20m, see `pipeline.py` for details)
resampled_bands = resample_all_bands_to_10m(clipped_bands)

# Reproject points into the same CRS as the resampled bands
af_points = points.to_crs(resampled_bands["B02"].rio.crs)

profile = profile_from_dataarray(resampled_bands["B02"])
band_arrays = {name: dataarray_to_numpy(da) for name, da in resampled_bands.items()}

In [ ]:
# STEP 3: Build focal buffer rasters
# NOTE: This will create 150 raster files in the `out_dir` that you specify

focal_paths = build_all_focal_buffers(
    band_arrays, pixel_size_m=10, out_dir="focal_rasters", profile=profile,
    n_jobs=-1, use_fft="auto"
)

In [ ]:
# STEP 4: Extract focal buffer values at observation points and build training table

table = build_training_table(af_points, focal_paths)

In [ ]:
# STEP 5-6: Train/test split and Random Forest training

rf_model, metrics = train_rf_model(table)

In [ ]:
# STEP 7: Predict a continuous surface across the full study area

predict_temperature_surface(
    rf_model, focal_paths,
    feature_order=table.drop(columns=["temperature_f"]).columns.tolist(),
    out_path=tif_output_path, 
    reference_profile=profile,
)

In [ ]:
# Open file to visualize raster output

with rasterio.open(f'./{tif_output_path}') as src:
    # Read the data (e.g., band 1)
    data = src.read(1, masked=True)
    
    # 2. Set up the matplotlib figure and axis
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # 3. Choose a colormap and establish min/max values
    cmap = 'RdYlBu_r'
    vmin, vmax = np.nanmin(data), np.nanmax(data)
    
    # 4. Plot the raster using rasterio's show()
    show(data, transform=src.transform, ax=ax, cmap=cmap, vmin=vmin, vmax=vmax)
    
    # 5. Add the colorbar
    fig.colorbar(
        cm.ScalarMappable(norm=colors.Normalize(vmin=vmin, vmax=vmax), cmap=cmap),
        ax=ax,
        label='Temperature (ºF)'
    )

    ax.set_xticks([])
    ax.set_yticks([])
    
    plt.title("")
    plt.show()